# How far to trust a velocity

EGMS gives a velocity for every 100 m cell where the radar found something stable to
measure. It does not say how far to trust it. `egms_client.reliability.trust` makes that
estimate from outside, from what the archive does give, and says why:

1. **coverage**: how many cells around the point have a value;
2. **fit**: how well the archive's model fits the cell's series (`rmse_ts`);
3. **the region**: the velocity against the median of its surroundings, because a still
   place is not at zero.

Three places from the benchmark: Rione Terra in Pozzuoli (it rises), the sea in front of it
(not measured), and a lone cell in the Foresta Umbra (a value, but not to trust).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The crops of the benchmark: small pieces of the real archive, in the repository.
# No key and no download: this notebook runs anywhere, and in CI.
DATA = Path("../benchmarks/data")
IMAGES = Path("../docs/images")  # the figure is saved there when the folder exists

In [ ]:
from egms_client.reliability import trust

PLACES = {
    "Rione Terra, Pozzuoli": ("campi-flegrei", 14.1205, 40.8225),
    "The gulf of Pozzuoli": ("campi-flegrei", 14.10, 40.80),
    "A lone cell, Foresta Umbra": ("foresta-umbra", 16.00776, 41.80508),
    "Gioia del Colle, the Murge": ("murge", 16.9236, 40.7997),
}
rows = []
for name, (crop, lon, lat) in PLACES.items():
    folder = DATA / crop
    r = trust([folder / "U.tif"], lon, lat, attributes=pd.read_parquet(folder / "l3.parquet"))
    rows.append(
        {
            "place": name,
            "velocity": r.velocity,
            "score": r.score,
            "reliable": r.reliable,
            "coverage": r.coverage,
            "regional median": r.regional_median,
            "relative": r.relative,
            "why": "; ".join(r.reasons),
        }
    )
pd.DataFrame(rows).set_index("place")

Read the table row by row:

- **Rione Terra**: +145.2 mm/year, score 1. Three cells in four around it have a value, and
  the series fits the model. Against its region it still rises by about 119 mm/year.
- **The gulf**: no velocity, score 0. *Not measured* is not zero: the radar sees no ground
  under the sea.
- **The lone cell**: it has a value, but 1 % of its neighbourhood has one. Score 0.03.
- **Gioia del Colle**: a still place. The velocity is close to the median of its region:
  that is what "still" looks like in EGMS.

## Coverage, cell by cell

`coverage_map` gives every cell the share of cells with a value within 500 m. The forest
is almost empty; the plain of Bologna, with its buildings and roads, is full.

In [ ]:
import rasterio

from egms_client.mosaic import mosaic
from egms_client.reliability import coverage_map

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, crop, title in zip(
    axes, ["foresta-umbra", "bologna"], ["Foresta Umbra", "Bologna"], strict=True
):
    tiff = DATA / crop / "U.tif"
    with rasterio.open(tiff) as src:
        left, bottom, right, top = src.bounds
    m = mosaic([tiff], minx=left, miny=bottom, maxx=right, maxy=top)
    cover = coverage_map(m.values)
    shown = ax.imshow(cover, vmin=0, vmax=1, cmap="viridis")
    have = float((~np.isnan(m.values)).mean())
    ax.set_title(f"{title}: {have:.1%} of the cells have a value")
    ax.set_axis_off()
fig.colorbar(shown, ax=axes, label="share of cells with a value within 500 m")
if IMAGES.exists():
    fig.savefig(IMAGES / "notebook-coverage.png", dpi=110, bbox_inches="tight")

## Still is not zero

East-west, the whole Murge box moves east by about 1.6 mm/year. Nothing slides there: it
is a regional offset, most likely the Adriatic plate against the reference of the L3
product. A velocity reads better against its region than against zero.

In [ ]:
murge = DATA / "murge" / "E.tif"
with rasterio.open(murge) as src:
    left, bottom, right, top = src.bounds
east = mosaic([murge], minx=left, miny=bottom, maxx=right, maxy=top).values
values = east[~np.isnan(east)]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(values, bins=np.arange(-4, 6, 0.25), color="#4292c6")
ax.axvline(0, color="k", lw=1)
ax.axvline(
    np.median(values), color="#cb181d", lw=2, label=f"median {np.median(values):+.1f} mm/year"
)
ax.set_xlabel("east-west velocity (mm/year), positive east")
ax.set_ylabel("cells")
ax.legend()
print(
    f"{values.size} cells; 5th to 95th percentile: {np.percentile(values, 5):+.1f} to {np.percentile(values, 95):+.1f}"
)

**What is not there.** Motion too fast for InSAR, north-south motion, anything after the
end of the release window (19 November 2025 for 2020-2025). The archive's own formal error,
`mean_velocity_std`, is reported but not trusted: it is at most 0.2 mm/year in a forest
where 99 % of the cells are empty.

*Data: EGMS, © European Union, Copernicus Land Monitoring Service.*